Code to create vapour pressure deficit (VPD) projections from daily tasmax and hursmin projections.

Equations saturation vapour pressure http://www.bom.gov.au/climate/how/newproducts/images/IDCJHC02_notes.txt

Vapour pressure = exp (1.8096 + (17.269425 * Dew_Point)/(237.3 + Dew_Point))

Saturated Vapour pressure = exp (1.8096 + (17.269425 * Air_Temperature)/(237.3 + Air_Temperature))

Relative Humidity = Vapour pressure / Saturated vapour pressure * 100

Rearrange the formulae to get:

Vapour pressure = rh * 0.0061094 * exp((17.652 * t)/(243.04 + t))

A nice explainer about the relevance of VPD to fire: https://blog.ucsusa.org/carly-phillips/what-is-vapor-pressure-deficit-vpd-and-what-is-its-connection-to-wildfires/

In [1]:
import dask
from dask.distributed import Client, wait
from dask import delayed

client = Client(n_workers=7, threads_per_worker=1) 
#client = Client()

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 7
Total threads: 7,Total memory: 63.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44511,Workers: 7
Dashboard: /proxy/8787/status,Total threads: 7
Started: Just now,Total memory: 63.00 GiB
Comm: tcp://127.0.0.1:45589,Total threads: 1
Dashboard: /proxy/39965/status,Memory: 9.00 GiB
Nanny: tcp://127.0.0.1:41609,


2025-05-09 10:54:47,594 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/g/data/ia39/ncra/bushfire/vpd/MPI-ESM1-2-HR/ssp370/r1i1p1f1/BARPA-R/v1-r1/day/ssp370_MPI-ESM1-2-HR_BARPA-R_gwl1.2_vpd.nc', lease_id='9e821944154f4585b2d847858eb4076c'. This can happen if the Lock or Semaphore timed out before.
2025-05-09 11:00:33,162 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/g/data/ia39/ncra/bushfire/vpd/MPI-ESM1-2-HR/ssp370/r1i1p1f1/BARPA-R/v1-r1/day/2nd_ssp370_MPI-ESM1-2-HR_BARPA-R_gwl1.5_vpd.nc', lease_id='c0e1c113a88d423f9dbb3d93a1e1b41b'. This can happen if the Lock or Semaphore timed out before.


In [2]:
#import all the stuff
from netCDF4 import Dataset
import xarray as xr
import numpy as np
import pandas as pd
from datetime import timedelta
import matplotlib.pyplot as plt
import glob
import sys
sys.path.append("/g/data/mn51/users/nb6195/project/gwls/")
import gwl

In [3]:
#function to compute VPD from tasmax and rh datasets
#input: are datasets of rh and tasmax
#output: datasets of vpd and monthly_mean_vpd

def vpd_calc(ds_rh, ds_tasmax):
#    vpd = (1 - ds_rh/100) * 0.61094 * np.exp((17.652 * ds_tasmax)/(243.04 + ds_tasmax)) #originally used funtion
    vpd = (1 - ds_rh/100) * 6.1094 * np.exp ((17.625 * ds_tasmax)/(243.04 + ds_tasmax)) #the formula that Blair uses from http://www.bom.gov.au/research/publications/cawcrreports/CTR_024.pdf
    monthly_mean_vpd = vpd.groupby('time.month').mean('time', keep_attrs=True)
    
    vpd.attrs = {
        'long_name': 'Daily maximum vapour dressure deficit computed from tasmax and hursmin',
        'standard_name': 'vpd',
        'units': 'hPa',
#        'regrid_method': 'bilinear'
    }
    ds_vpd = xr.Dataset({'vpd' : vpd})
    ds_rh.close()
    ds_tasmax.close()
    
    monthly_mean_vpd.attrs = {
        'long_name': 'Monthly mean vapour dressure deficit computed from tasmax and hursmin',
        'standard_name': 'monthly_mean_vpd',
        'units': 'hPa',
    }
    ds_monthly_mean_vpd = xr.Dataset({'monthly_mean_vpd' : monthly_mean_vpd})
    return ds_vpd, ds_monthly_mean_vpd

BOM to-do: 
- CESM2
- CMCC-ESM2
- NorESM2-MM

BOM redone: 
- MPI-ESM1-2-HR (r1i1p1f1)
- EC-Earth3 (r1i1p1f1)
- ACCESS-CM2 (r4i1p1f1)
- ACCESS-ESM1-5 (r6i1p1f1)

CSIRO todo: 
- CNRM-ESM2-1 (r1i1p1f2)
- CESM2
- CMCC-ESM2
- NorESM2-MM

CSIRO redone:
- EC-Earth3
- ACCESS-CM2 (r4i1p1f1)
- ACCESS-ESM1-5 (r6i1p1f1)

In [4]:
#Set parameters
CMIP='CMIP6'
#AGENCY = 'CSIRO' 
#RCM = 'CCAM-v2203-SN'
AGENCY = 'BOM' 
RCM = 'BARPA-R'

#GCM = 'ACCESS-CM2' #ensemble = 'r4i1p1f1' #Done
#GCM = 'ACCESS-ESM1-5' #ensemble = 'r6i1p1f1' #Done
#GCM = 'EC-Earth3' #ensemble = 'r1i1p1f1' #Done
GCM = 'MPI-ESM1-2-HR' 
ensemble = 'r1i1p1f1' #BOM done, no CSIRO
#GCM = 'CESM2' #ensemble = 'r11i1p1f1' #Done
#GCM = 'CMCC-ESM2' #ensemble = 'r1i1p1f1' #Done
#GCM = 'NorESM2-MM' #ensemble = 'r1i1p1f1' #Done
#GCM = 'CNRM-ESM2-1' #ensemble = 'r1i1p1f2' #CSIRO Done, no BOM


#pathway = 'ssp126'
pathway = 'ssp370'

output_dir = '/g/data/ia39/ncra/bushfire/vpd/'
output_dir_mm = '/g/data/ia39/ncra/bushfire/vpd/monthly_mean/'

In [5]:
#read in RCM files
var1 = 'tasmax'

#ddir = f"/g/data/ia39/australian-climate-service/release/CORDEX/output-Adjust/{CMIP}/bias-adjusted-input/AUST-05i/{AGENCY}/{GCM}/{pathway}/{ensemble}/{RCM}/v1-r1/day/{var1}/v20241216"
#infiles1=glob.glob(ddir+f'/{var1}_AUST-05i_{GCM}_{pathway}_{ensemble}_{AGENCY}_{RCM}_v1-r1_day_*.nc')
#tasmax_master_ds = xr.open_mfdataset(infiles1)

ddir = f"/g/data/kj66/CORDEX/output/{CMIP}/DD/AUST-05i/{AGENCY}/{GCM}"
infiles1a=glob.glob(ddir+f'/historical/{ensemble}/{RCM}/v1-r1/day/{var1}/v20241216//{var1}_AUST-05i_{GCM}_historical_{ensemble}_{AGENCY}_{RCM}_v1-r1_day_*.nc')
infiles1b=glob.glob(ddir+f'/{pathway}/{ensemble}/{RCM}/v1-r1/day/{var1}/v20241216/{var1}_AUST-05i_{GCM}_{pathway}_{ensemble}_{AGENCY}_{RCM}_v1-r1_day_*.nc')
tasmax_master_ds = xr.open_mfdataset(infiles1a + infiles1b)

In [6]:
var2 = 'hursmin'

#ddir = f"/g/data/ia39/australian-climate-service/release/CORDEX/output-Adjust/{CMIP}/bias-adjusted-input/AUST-05i/{AGENCY}/{GCM}/{pathway}/{ensemble}/{RCM}/v1-r1/day/{var2}/v20241216"
#infiles2=glob.glob(ddir+f'/{var2}_AUST-05i_{GCM}_{pathway}_{ensemble}_{AGENCY}_{RCM}_v1-r1_day_*.nc')
#hursmin_master_ds = xr.open_mfdataset(infiles2)

#hursmin_AUST-05i_CESM2_ssp370_r11i1p1f1_BOM_BARPA-R_v1-r1_day_20760101-20761231.nc

ddir2 = f"/g/data/kj66/CORDEX/output/{CMIP}/DD/AUST-05i/{AGENCY}/{GCM}"
infiles2a=glob.glob(ddir+f'/historical/{ensemble}/{RCM}/v1-r1/day/{var2}/v20241216/{var2}_AUST-05i_{GCM}_historical_{ensemble}_{AGENCY}_{RCM}_v1-r1_day_*.nc')
infiles2b=glob.glob(ddir+f'/{pathway}/{ensemble}/{RCM}/v1-r1/day/{var2}/v20241216/{var2}_AUST-05i_{GCM}_{pathway}_{ensemble}_{AGENCY}_{RCM}_v1-r1_day_*.nc')
hursmin_master_ds = xr.open_mfdataset(infiles2a + infiles2b)

Need to check that the time of tasmax and the time of hursmin align, else a shift in the time will be required for the calculation to work - uncomment code 2 cells below if the shift is needed

In [7]:
#tasmax_master_ds

In [8]:
#hursmin_master_ds

In [9]:
#plt.imshow(hursmin_master_ds['hursmin'][50], origin='lower')
#plt.colorbar()

In [21]:
#Extract time period corresponding to the chosen GWL for tasmax and rh
chosen_gwl = '1.5'

gwl_tasmax = gwl.get_GWL_timeslice(tasmax_master_ds,CMIP,GCM,ensemble,pathway,GWL=chosen_gwl)[var1]
gwl_rh = gwl.get_GWL_timeslice(hursmin_master_ds,CMIP,GCM,ensemble,pathway,GWL=chosen_gwl)[var2]
#gwl_rh = gwl_rh.assign_coords(time = pd.to_datetime(gwl_rh.time) + timedelta(hours = 12)) #needed

In [22]:
#gwl_rh.indexes['time'].to_datetimeindex()

#gwl_rh = gwl_rh.assign_coords(time = gwl_rh.indexes['time'].to_datetimeindex() + timedelta(hours = 12))
#gwl_rh['month'] = gwl_rh['time'].dt.month
#gwl_tasmax['month'] = gwl_tasmax['time'].dt.month

In [23]:
#Check that gwl_tasmax and gwl_rh have the same calendar
#gwl_tasmax.time.dt.calendar
#gwl_rh.time.dt.calendar

In [24]:
#Create the datasets for vpd and monthly mean vpd
gwl_vpd, monthly_mean_vpd = vpd_calc(gwl_rh, gwl_tasmax)

In [26]:
#print half files for gwl_vpd
#halves = ['1st', '2nd']
halves = ['2nd']

for half in halves:
    print(half)
    if half == '1st':
        gwl_vpd_print = gwl_vpd['vpd'][0:2737]
    else: 
        gwl_vpd_print = gwl_vpd['vpd'][2737:]
        
    file_name_vpd = half + '_' + pathway + '_' + GCM + '_' + RCM + '_gwl' + chosen_gwl + '_vpd.nc'
    output_file_location = output_dir + GCM + '/' + pathway + '/' + ensemble + '/' + RCM + '/v1-r1/day/' + file_name_vpd
    gwl_vpd_print.to_netcdf(output_file_location, engine='netcdf4')
        
    print('half done')
    print(output_file_location)
    

2nd
half done
/g/data/ia39/ncra/bushfire/vpd/MPI-ESM1-2-HR/ssp370/r1i1p1f1/BARPA-R/v1-r1/day/2nd_ssp370_MPI-ESM1-2-HR_BARPA-R_gwl1.5_vpd.nc


In [27]:
#print vpd to an external file
#this is too big, has been replaced by printing half files above
#file_name_vpd = pathway + '_' + GCM + '_' + RCM + '_gwl' + chosen_gwl + '_vpd.nc' 
#output_file_location = output_dir + GCM + '/' + pathway + '/' + ensemble + '/' + RCM + '/v1-r1/day/' + file_name_vpd
#gwl_vpd.to_netcdf(output_file_location, engine='netcdf4')
#print(output_file_location)

In [28]:
#print monthly mean ds to external file

file_name_mean = 'gwl' + chosen_gwl + '_monthly_mean_vpd_' + GCM + '_' + RCM + '_' + pathway + '_' + ensemble + '.nc' 
output_file_location = output_dir_mm + file_name_mean
monthly_mean_vpd.to_netcdf(output_file_location, engine='netcdf4')
print(output_file_location)

/g/data/ia39/ncra/bushfire/vpd/monthly_mean/gwl1.5_monthly_mean_vpd_MPI-ESM1-2-HR_BARPA-R_ssp370_r1i1p1f1.nc
